In [ ]:



import pyarrow
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats



# Path to directory
DATA_PATH = Path(r"C:\Users\benel\OneDrive\Desktop\Python\Thesis_xyz")
output_dir_metrics = DATA_PATH / "results" / "metrics" / "random_forest"
output_dir_plots = DATA_PATH / "results" / "plots" / "random_forest"
output_dir_metrics.mkdir(parents=True, exist_ok=True)
output_dir_plots.mkdir(parents=True, exist_ok=True)
DATA_DIR = DATA_PATH / "results" / "data" / "random_forest"

# Date Range
start_date = "1998-01-01"
end_date = "2025-12-31"
risk_free_rate = 0.0

FREQUENCIES = {
    'Yearly': pd.DateOffset(years=1),
    'Semi-Annual': pd.DateOffset(months=6),
    'Quarterly': pd.DateOffset(months=3),
    'Monthly': pd.DateOffset(months=1)
}

# (label, short_key, start_date, trough_date, end_date)
# Dates are S&P 500 (benchmark) defined crisis windows
CRISIS_PERIODS = [
    ('Dotcom Crash',       'dotcom',  '2000-03-23', '2002-10-09', '2007-05-31'),
    ('GFC',                'gfc',     '2007-10-09', '2009-03-09', '2013-03-28'),
    ('Monetary Policy',    'mon_pol', '2018-09-21', '2018-12-24', '2019-04-23'),
    ('COVID-19',           'covid19', '2020-02-19', '2020-03-23', '2020-08-12'),
    ('Russia/Ukraine',     'russia',  '2022-01-03', '2022-10-12', '2024-01-19'),
    ('Trade Policy Shock', 'trade',   '2025-02-19', '2025-04-08', '2025-06-26'),
]

CRISIS_METRIC_LABELS = [
    ('max_drawdown',            'Max Drawdown (%)',               '{:.2%}'),
    ('days_to_trough',          'Days: Peak → Trough',           '{:.0f}'),
    ('days_trough_to_recovery', 'Days: Trough → Breakeven',      '{:.0f}'),
    ('days_peak_to_breakeven',  'Days: Peak → Breakeven (Total)','{:.0f}'),
    ('crisis_cum_return',       'Cumulative Return (Crisis)',     '{:.2%}'),
    ('crisis_ann_return',       'Annualized Return (Crisis)',     '{:.2%}'),
    ('crisis_ann_volatility',   'Annualized Volatility (Crisis)', '{:.2%}'),
    ('crisis_sharpe',           'Sharpe Ratio (Crisis)',          '{:.3f}'),
    ('crisis_sortino',          'Sortino Ratio (Crisis)',         '{:.3f}'),
]


# --------
# Create the Metrics
# --------
# Helpers for Frequency
TRADING_DAYS_PER_YEAR = 252
MONTHS_PER_YEAR = 12

def _annualized_factor(freq: str) -> int:
    """Return the annualization factor based on return frequency"""
    if freq == 'D':
        return TRADING_DAYS_PER_YEAR
    elif freq == 'M':
        return MONTHS_PER_YEAR
    else:
        raise ValueError(f"freq must be 'D' or 'M', got '{freq}'")

# Return Metrics
def cumulative_return(log_returns_per_day: pd.Series) -> float:
    """Total cumulative return over the full period"""
    total_log_return = log_returns_per_day.sum()
    return np.exp(total_log_return) - 1

def annualized_return(log_returns_per_day: pd.Series, freq: str = 'D') -> float:
    """Annualized return, accounting for compounding"""
    total_log_return = log_returns_per_day.sum()
    n = len(log_returns_per_day)
    ann_f = _annualized_factor(freq)
    annual_log_return = (total_log_return / n) * ann_f
    return np.exp(annual_log_return) - 1

# Risk Metrics
def annualized_volatility(log_returns_per_day: pd.Series, freq: str = 'D') -> float:
    """Annualized standard deviation of returns"""
    std_dev = log_returns_per_day.std(ddof=1)
    ann_f = _annualized_factor(freq)
    ann_volatility = std_dev * np.sqrt(ann_f)
    return ann_volatility

def maximum_drawdown(price_series: pd.Series) -> float:
    """Maximum Drawdown"""
    rolling_max = price_series.cummax()
    drawdowns = (price_series / rolling_max) - 1
    return drawdowns.min()

def drawdown_series(price_series: pd.Series) -> pd.Series:
    """Return the full drawdown time series"""
    rolling_max = price_series.cummax()
    drawdown_series = (price_series / rolling_max) - 1
    return drawdown_series

def max_drawdown_duration(price_series: pd.Series) -> int:
    """Duration of Drawdown"""
    rolling_max = price_series.cummax()
    is_at_peak = price_series >= rolling_max
    underwater_groups = is_at_peak.cumsum()
    durations = price_series.groupby(underwater_groups).cumcount()
    return durations.max()

def recovery_duration(price_series: pd.Series) -> int:
    """Duration of Recovery"""
    rolling_max = price_series.cummax()
    drawdown = (price_series / rolling_max) - 1
    mdd_date = drawdown.idxmin()
    peak_at_mdd = rolling_max.loc[mdd_date]
    recovery_series = price_series.loc[mdd_date:]
    recovered_hits = recovery_series[recovery_series >= peak_at_mdd]
    if recovered_hits.empty:
        return len(recovery_series)
    recovery_date = recovered_hits.index[0]
    return len(price_series.loc[mdd_date:recovery_date]) - 1

def value_at_risk(log_returns_per_day: pd.Series, confidence_level: float = 0.95) -> float:
    """Historical Value at Risk (VaR 95%)"""
    return log_returns_per_day.quantile(1 - confidence_level)

def conditional_value_at_risk(log_returns_per_day: pd.Series, confidence_level: float = 0.95) -> float:
    """Historical Conditional Value at Risk (CVaR 95%)"""
    var_threshold = value_at_risk(log_returns_per_day, confidence_level)
    return log_returns_per_day[log_returns_per_day <= var_threshold].mean()

def beta(portfolio_returns: pd.Series, benchmark_returns: pd.Series) -> float:
    """
    Beta — sensitivity of portfolio returns to benchmark returns.
    Beta > 1: more volatile than benchmark.
    """
    aligned_p, aligned_b = portfolio_returns.align(benchmark_returns, join='inner')
    cov   = np.cov(aligned_p, aligned_b)
    return cov[0, 1] / cov[1, 1]

def alpha(portfolio_returns: pd.Series, benchmark_returns: pd.Series, risk_free_rate:    float = 0.02, freq:              str   = 'M',) -> float:
    """Excess Return"""
    b       = beta(portfolio_returns, benchmark_returns)
    ann_p   = annualized_return(portfolio_returns, freq)
    ann_bm  = annualized_return(benchmark_returns, freq)
    return ann_p - (risk_free_rate + b * (ann_bm - risk_free_rate))

# Risk Adjusted Metrics
def sharpe_ratio(log_returns_per_day: pd.Series, risk_free_rate, freq: str = 'D') -> float:
    """Annualized Sharpe Ratio"""
    ann_ret = annualized_return(log_returns_per_day, freq)
    ann_vol = annualized_volatility(log_returns_per_day, freq)
    return (ann_ret - risk_free_rate) / ann_vol

def sortino_ratio(log_returns_per_day: pd.Series, risk_free_rate, freq: str = 'D') -> float:
    """Annualized Sortino Ratio"""
    ann_ret = annualized_return(log_returns_per_day, freq)
    ann_f = _annualized_factor(freq)
    downside_sq = np.minimum(log_returns_per_day, 0) ** 2
    downside_vol = np.sqrt(downside_sq.mean()) * np.sqrt(ann_f)
    return (ann_ret - risk_free_rate) / downside_vol

def calmar_ratio(log_returns_per_day: pd.Series, price_series: pd.Series, freq: str = 'D') -> float:
    """Calmar Ratio (Return/Max Drawdown)"""
    ann_ret = annualized_return(log_returns_per_day, freq)
    mdd = abs(maximum_drawdown(price_series))
    return ann_ret / mdd if mdd != 0 else np.nan

def omega_ratio(log_returns_per_day: pd.Series, threshold: float = 0.0) -> float:
    """Omega Ratio (Probability-weighted gains vs losses)"""
    gains = log_returns_per_day[log_returns_per_day > threshold].sum()
    losses = abs(log_returns_per_day[log_returns_per_day < threshold].sum())
    return gains / losses if losses != 0 else np.inf

def named_crisis_metrics(log_returns: pd.Series, price_series: pd.Series, freq: str = 'D',) -> dict:
    """Compute peak→trough→recovery metrics for each crisis in CRISIS_PERIODS.
    Trough is the worst point within the crisis window; peak is the all-time
    high before that trough; recovery is the first date (anywhere in the series
    after the trough) where price returns to the pre-trough peak level."""
    results = {}
    full_dd = (price_series / price_series.cummax()) - 1

    for _, crisis_key, window_start, _, window_end in CRISIS_PERIODS:
        window_dd = full_dd.loc[window_start:window_end]
        if window_dd.empty:
            continue

        # Trough: worst drawdown within the crisis window (relative loss, not lowest price)
        trough_date = window_dd.idxmin()
        trough_val  = price_series.loc[trough_date]

        # Peak: the running all-time high just before the trough (flexible — data-driven)
        peak_date = price_series.loc[:trough_date].idxmax()
        peak_val  = price_series.loc[peak_date]

        max_dd = (trough_val / peak_val) - 1
        days_to_trough = (pd.Timestamp(trough_date) - pd.Timestamp(peak_date)).days

        # Recovery: first date after trough where price >= peak (search full series)
        post_trough    = price_series.loc[trough_date:]
        recovered_hits = post_trough[post_trough >= peak_val]
        if not recovered_hits.empty:
            recovery_date            = recovered_hits.index[0]
            days_trough_to_recovery  = (pd.Timestamp(recovery_date) - pd.Timestamp(trough_date)).days
            days_peak_to_breakeven   = (pd.Timestamp(recovery_date) - pd.Timestamp(peak_date)).days
        else:
            days_trough_to_recovery  = np.nan
            days_peak_to_breakeven   = np.nan
            recovery_date            = price_series.index[-1]

        # Full crisis window returns (start → end)
        crisis_ret = log_returns.loc[window_start:window_end]

        def _crisis_sortino(lr):
            ann_ret = annualized_return(lr, freq)
            ds_vol  = np.sqrt((np.minimum(lr, 0) ** 2).mean()) * np.sqrt(_annualized_factor(freq))
            return ann_ret / ds_vol if ds_vol != 0 else np.nan

        k = crisis_key
        results[f'{k}_max_drawdown']            = max_dd
        results[f'{k}_days_to_trough']          = days_to_trough
        results[f'{k}_days_trough_to_recovery'] = days_trough_to_recovery
        results[f'{k}_days_peak_to_breakeven']  = days_peak_to_breakeven
        results[f'{k}_crisis_cum_return']       = cumulative_return(crisis_ret)       if len(crisis_ret) > 0 else np.nan
        results[f'{k}_crisis_ann_return']       = annualized_return(crisis_ret, freq) if len(crisis_ret) > 1 else np.nan
        results[f'{k}_crisis_ann_volatility']   = annualized_volatility(crisis_ret, freq) if len(crisis_ret) > 1 else np.nan
        results[f'{k}_crisis_sharpe']           = sharpe_ratio(crisis_ret, 0.0, freq) if len(crisis_ret) > 1 else np.nan
        results[f'{k}_crisis_sortino']          = _crisis_sortino(crisis_ret)         if len(crisis_ret) > 1 else np.nan

    return results

def information_ratio(portfolio_returns: pd.Series, benchmark_returns: pd.Series, freq: str = 'M',) -> float:
    """Information Ratio — measures payoff between active returns and
    tracking error (volatility of active returns vs benchmark)"""
    ann_f          = _annualized_factor(freq)
    aligned_p, aligned_b = portfolio_returns.align(benchmark_returns, join='inner')
    active_returns = aligned_p - aligned_b
    if active_returns.std() == 0:
        return np.nan
    return (active_returns.mean() / active_returns.std()) * np.sqrt(ann_f)

def modified_information_ratio(portfolio_returns: pd.Series, benchmark_returns: pd.Series, freq: str = 'M',) -> float:
    """Modified Information Ratio — uses maximum drawdown instead of tracking error to provide a more synthetic view of active risk."""
    aligned_p, aligned_b = portfolio_returns.align(benchmark_returns, join='inner')
    active_returns = aligned_p - aligned_b
    ann_active     = annualized_return(active_returns, freq)
    max_dd         = maximum_drawdown(active_returns)
    if max_dd == 0:
        return np.nan
    return ann_active / abs(max_dd)

def treynor_ratio(portfolio_returns: pd.Series, benchmark_returns: pd.Series, risk_free_rate:    float = 0.02, freq:str   = 'M',) -> float:
    """Treynor Ratio — excess return per unit of systematic risk (beta). Useful for comparing portfolios with different betas"""
    ann_f   = _annualized_factor(freq)
    b       = beta(portfolio_returns, benchmark_returns)
    ann_ret = annualized_return(portfolio_returns, freq)
    if b == 0:
        return np.nan
    return (ann_ret - risk_free_rate) / b

# Calculate the Metrics
def compute_all_metrics(portfolio_log_returns: pd.Series, price_series: pd.Series, benchmark_log_returns: pd.Series, risk_free_rate, freq: str = 'D') -> dict:
    """Compute all portfolio evaluation metrics"""
    results = {}
    log_ret, bench_ret = portfolio_log_returns.align(benchmark_log_returns, join='inner')
    price_aligned = price_series.loc[log_ret.index]
    results['cumulative_return']      = cumulative_return(log_ret)
    results['annualized_return']      = annualized_return(log_ret, freq)
    results['benchmark_cum_return']   = cumulative_return(bench_ret)
    results['benchmark_ann_return']   = annualized_return(bench_ret, freq)
    results['annualized_volatility']  = annualized_volatility(log_ret, freq)
    results['maximum_drawdown']       = maximum_drawdown(price_aligned)
    results['dd_duration_to_trough']  = max_drawdown_duration(price_aligned)
    results['recovery_duration']      = recovery_duration(price_aligned)
    results['var_95']                 = value_at_risk(log_ret, 0.95)
    results['cvar_95']                = conditional_value_at_risk(log_ret, 0.95)
    results['sharpe']                 = sharpe_ratio(log_ret, risk_free_rate, freq)
    results['sortino']                = sortino_ratio(log_ret, risk_free_rate, freq)
    results['calmar']                 = calmar_ratio(log_ret, price_aligned, freq)
    results['omega']                  = omega_ratio(log_ret)
    results['beta']                   = beta(log_ret, bench_ret)
    results['alpha']                  = alpha(log_ret, bench_ret, risk_free_rate, freq)
    results['information_ratio']      = information_ratio(log_ret, bench_ret, freq)
    crisis_results = named_crisis_metrics(log_ret, price_aligned, freq)
    results.update(crisis_results)
    return results

# Create DataFrame
def metrics_to_dataframe(results: dict) -> pd.DataFrame:
    """Convert results dict to a clean formatted DataFrame for display/export."""
    labels = {
        'cumulative_return'          : ('Returns',   'Cumulative Return',           '{:.2%}'),
        'annualized_return'          : ('Returns',   'Annualized Return',           '{:.2%}'),
        'benchmark_cum_return'       : ('Benchmark', 'Benchmark Cumulative Return', '{:.2%}'),
        'benchmark_ann_return'       : ('Benchmark', 'Benchmark Annualized Return', '{:.2%}'),
        'annualized_volatility'      : ('Risk',      'Annualized Volatility',       '{:.2%}'),
        'maximum_drawdown'           : ('Risk',      'Maximum Drawdown',            '{:.2%}'),
        'var_95'                     : ('Risk',      'Value at Risk (95%)',         '{:.2%}'),
        'cvar_95'                    : ('Risk',      'CVaR / Expected Shortfall',   '{:.2%}'),
        'sharpe'                     : ('Ratios',    'Sharpe Ratio',                '{:.3f}'),
        'sortino'                    : ('Ratios',    'Sortino Ratio',               '{:.3f}'),
        'calmar'                     : ('Ratios',    'Calmar Ratio',                '{:.3f}'),
        'omega'                      : ('Ratios',    'Omega Ratio',                 '{:.3f}'),
        'beta'                       : ('Benchmark', 'Beta',                        '{:.3f}'),
        'alpha'                      : ('Benchmark', 'Alpha (Annualized)',          '{:.2%}'),
        'information_ratio'          : ('Benchmark', 'Information Ratio',           '{:.3f}'),
    }

    # Build crisis labels dynamically from CRISIS_PERIODS + CRISIS_METRIC_LABELS
    crisis_labels = {}
    for crisis_name, crisis_key, _, _, _ in CRISIS_PERIODS:
        for metric_suffix, metric_label, fmt in CRISIS_METRIC_LABELS:
            crisis_labels[f'{crisis_key}_{metric_suffix}'] = (crisis_name, metric_label, fmt)

    def _fmt_value(value, fmt):
        try:
            return fmt.format(value) if not (isinstance(value, float) and np.isnan(value)) else 'N/A'
        except Exception:
            return str(value)

    rows = []
    for key, value in results.items():
        if key in labels:
            category, name, fmt = labels[key]
            rows.append({'Category': category, 'Metric': name, 'Value': _fmt_value(value, fmt)})
        elif key in crisis_labels:
            category, name, fmt = crisis_labels[key]
            rows.append({'Category': category, 'Metric': name, 'Value': _fmt_value(value, fmt)})

    return pd.DataFrame(rows)

# --------
# Create the Plots
# --------
def _find_drawdown_episodes(df, threshold):
    """Find drawdown episodes where dd breaches -threshold.
    Groups by full underwater periods (dd < 0) so a single drawdown that
    temporarily recovers above the threshold is still counted as ONE episode.
    Returns list of dicts: peak_date, trough_date, recovery_date,
    dd_pct, days_to_trough, days_to_recovery, recovered."""
    cum = df['cum_return']
    dd  = df['drawdown']
    episodes = []

    is_underwater = dd < 0
    if not is_underwater.any():
        return episodes

    group_ids = (is_underwater != is_underwater.shift()).cumsum()

    for _, grp in df[is_underwater].groupby(group_ids[is_underwater]):
        # Skip episodes that never breach the threshold
        if grp['drawdown'].min() > -threshold:
            continue
        trough_date = grp['drawdown'].idxmin()
        dd_pct      = dd.loc[trough_date]

        group_start = grp.index[0]
        peak_date   = cum.loc[:group_start].idxmax()
        peak_val    = cum.loc[peak_date]

        post        = cum.loc[trough_date:]
        hits        = post[post >= peak_val]
        if not hits.empty:
            recovery_date = hits.index[0]
            recovered     = True
        else:
            recovery_date = cum.index[-1]
            recovered     = False

        episodes.append({
            'peak_date'        : peak_date,
            'trough_date'      : trough_date,
            'recovery_date'    : recovery_date,
            'dd_pct'           : dd_pct,
            'days_to_trough'   : (pd.Timestamp(trough_date)   - pd.Timestamp(peak_date)).days,
            'days_to_recovery' : (pd.Timestamp(recovery_date) - pd.Timestamp(trough_date)).days,
            'recovered'        : recovered,
        })
    return episodes

def _build_drawdown_table(episodes):
    """Build overall stats + quartile breakdown table data from drawdown episodes.
    Returns (header_values, cell_values) ready for go.Table"""
    if not episodes:
        return [], []

    dds  = [abs(ep['dd_pct'])           for ep in episodes]
    durs = [ep['days_to_trough']        for ep in episodes]
    recs = [ep['days_to_recovery']      for ep in episodes]

    n = len(episodes)

    # Sort by severity (worst first) and split into quartiles
    sorted_eps = sorted(episodes, key=lambda e: e['dd_pct'])   # most negative first
    q_size     = max(1, n // 4)
    quartiles  = [sorted_eps[i * q_size:(i + 1) * q_size] for i in range(4)]
    # put any remainder into Q4
    if len(sorted_eps) > 4 * q_size:
        quartiles[3] += sorted_eps[4 * q_size:]

    def stats(eps_list):
        if not eps_list:
            return "—", "—", "—", 0
        d  = [abs(e['dd_pct'])      for e in eps_list]
        du = [e['days_to_trough']   for e in eps_list]
        re = [e['days_to_recovery'] for e in eps_list]
        return (
            f"{np.mean(d):.1%}",
            f"{np.mean(du):.0f}d",
            f"{np.mean(re):.0f}d",
            len(eps_list),
        )

    rows = []
    # Overall
    rows.append(("All episodes", f"{n}", f"{np.mean(dds):.1%}", f"{np.mean(durs):.0f}d", f"{np.mean(recs):.0f}d"))
    # Quartiles: Q1 = most severe
    labels = ["Q1 — most severe (top 25%)", "Q2", "Q3", "Q4 — mildest (bottom 25%)"]
    for label, q_eps in zip(labels, quartiles):
        avg_dd, avg_dur, avg_rec, cnt = stats(q_eps)
        rows.append((label, str(cnt), avg_dd, avg_dur, avg_rec))

    headers = ["Quartile", "Count", "Avg DD %", "Avg Duration to Trough", "Avg Recovery Duration"]
    cells   = list(zip(*rows))   # transpose
    return headers, cells

def generate_dynamic_benchmark_report(portfolio_df, title="Portfolio Performance Analysis", threshold=0.05) -> str:
    """Creates a white-themed HTML report: cumulative return + drawdown table, underwater, return distribution."""
    df = portfolio_df.copy()
    df.index = pd.to_datetime(df.index)
    # Adaptation: markowitz data has log_return + cumulative_value instead of returns_per_day + price
    if 'returns_per_day' not in df.columns:
        df['returns_per_day'] = np.exp(df['log_return']) - 1
    if 'cum_return' not in df.columns:
        df['cum_return'] = df['cumulative_value'] if 'cumulative_value' in df.columns else (1 + df['returns_per_day']).cumprod()
    df['drawdown'] = (df['cum_return'] / df['cum_return'].cummax()) - 1

    episodes = _find_drawdown_episodes(df, threshold)
    worst    = min(episodes, key=lambda e: e['dd_pct']) if episodes else None

    # ── Named crisis episodes: use explicit S&P 500 dates directly ────────────
    _named_eps = []
    for _cname, _, _c_start, _c_trough, _c_end in CRISIS_PERIODS:
        _idx = df.index
        _pd  = _idx[_idx >= pd.Timestamp(_c_start)][0]   if any(_idx >= pd.Timestamp(_c_start))  else None
        _td  = _idx[_idx >= pd.Timestamp(_c_trough)][0]  if any(_idx >= pd.Timestamp(_c_trough)) else None
        _rd  = _idx[_idx >= pd.Timestamp(_c_end)][0]     if any(_idx >= pd.Timestamp(_c_end))    else _idx[-1]
        if _pd is None or _td is None:
            continue
        _pv = df['cum_return'].loc[_pd]
        _tv = df['cum_return'].loc[_td]
        _named_eps.append({'name': _cname, 'peak_date': _pd, 'trough_date': _td,
                           'recovery_date': _rd, 'dd_pct': (_tv / _pv) - 1})

    # ── Colours ───────────────────────────────────────────────────────────────
    BG        = "#FFFFFF"
    PANEL     = "#F8F9FA"
    GRID      = "#E9ECEF"
    LINE_BLUE = "#1D6FA4"
    RED       = "#DC2626"
    GREEN     = "#16A34A"
    TEXT      = "#1E293B"
    SUBTEXT   = "#64748B"
    TBL_HDR   = "#1D6FA4"
    TBL_ROW_B = "#FFFFFF"
    Q1_COLOR  = "#FEE2E2"   # lightest red
    Q2_COLOR  = "#FEF9C3"
    Q3_COLOR  = "#DCFCE7"
    Q4_COLOR  = "#DBEAFE"   # lightest blue

    # ── Layout: 4 rows (cum return, table, underwater, distribution) ───────────
    # ── Prepare annual + monthly return data ──────────────────────────────────
    df_annual  = df['returns_per_day'].resample('YE').apply(lambda r: (1 + r).prod() - 1)
    df_annual.index = df_annual.index.year

    monthly    = df['returns_per_day'].resample('ME').apply(lambda r: (1 + r).prod() - 1)
    monthly_df = monthly.to_frame('ret')
    monthly_df['year']  = monthly_df.index.year
    monthly_df['month'] = monthly_df.index.month
    heat_pivot = monthly_df.pivot(index='year', columns='month', values='ret')
    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    heat_pivot.columns = [month_names[m - 1] for m in heat_pivot.columns]
    heat_z     = heat_pivot.values
    heat_years = [str(y) for y in heat_pivot.index]

    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.05,
        row_heights=[0.28, 0.12, 0.14, 0.16, 0.14, 0.32],
        subplot_titles=[
            f"Cumulative Return  —  drawdown episodes > {threshold:.0%} highlighted",
            f"Drawdown Episode Summary (threshold {threshold:.0%})",
            "Underwater Plot (Drawdown %)",
            "Daily Return Distribution",
            "Annual Returns",
            "Monthly Returns Heatmap",
        ],
        specs=[
            [{"type": "xy"}],
            [{"type": "table"}],
            [{"type": "xy"}],
            [{"type": "xy"}],
            [{"type": "xy"}],
            [{"type": "xy"}],
        ],
    )

    # ── Chart 1: Cumulative Return ─────────────────────────────────────────────
    fig.add_trace(
        go.Scatter(
            x=df.index, y=df['cum_return'],
            mode='lines',
            line=dict(color=LINE_BLUE, width=2),
            name="Cum. Return",
            hovertemplate="%{x|%b %d, %Y}<br>Cum. Return: %{y:.1%}<extra></extra>",
        ),
        row=1, col=1,
    )

    for ep in episodes:
        is_worst    = worst is not None and ep['trough_date'] == worst['trough_date']
        red_alpha   = 0.25 if is_worst else 0.12
        green_alpha = 0.20 if is_worst else 0.10

        fig.add_vrect(
            x0=ep['peak_date'], x1=ep['trough_date'],
            fillcolor=RED, opacity=red_alpha,
            layer="below", line_width=0,
            row=1, col=1,
        )
        fig.add_vrect(
            x0=ep['trough_date'], x1=ep['recovery_date'],
            fillcolor=GREEN, opacity=green_alpha,
            layer="below", line_width=0,
            row=1, col=1,
        )

        rec_label = f"↑ {ep['days_to_recovery']}d" if ep['recovered'] else "↑ n/a"
        prefix    = "<b>★ Worst</b><br>" if is_worst else ""
        ay_offset = -70 if is_worst else -48
        ann_color  = "#991B1B" if is_worst else RED

        fig.add_annotation(
            x=ep['trough_date'],
            y=df.loc[ep['trough_date'], 'cum_return'],
            xref="x", yref="y",
            text=(
                f"{prefix}"
                f"<b>{ep['dd_pct']:.1%}</b><br>"
                f"↓ {ep['days_to_trough']}d  {rec_label}"
            ),
            showarrow=True,
            arrowhead=2,
            arrowcolor=ann_color,
            arrowwidth=1.2,
            arrowsize=0.9,
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor=ann_color,
            borderwidth=1,
            font=dict(size=9, color=TEXT),
            ax=0, ay=ay_offset,
            row=1, col=1,
        )

    # ── Named crisis highlights: stronger bands with black border ─────────────
    _y_top = float(df['cum_return'].max())
    for _ep in _named_eps:
        fig.add_vrect(
            x0=_ep['peak_date'], x1=_ep['trough_date'],
            fillcolor='rgba(220,38,38,0.40)', opacity=1,
            layer='below', line_width=1.5, line_color='black',
            row=1, col=1,
        )
        fig.add_vrect(
            x0=_ep['trough_date'], x1=_ep['recovery_date'],
            fillcolor='rgba(22,163,74,0.28)', opacity=1,
            layer='below', line_width=1.5, line_color='black',
            row=1, col=1,
        )
        fig.add_annotation(
            x=_ep['peak_date'], y=_y_top,
            xref='x', yref='y',
            text=f"<b>{_ep['name']}</b>",
            showarrow=False, xanchor='left', yanchor='top',
            font=dict(size=9, color='black', family='Inter, Arial'),
            bgcolor='rgba(255,255,255,0.85)',
            bordercolor='black', borderwidth=1,
        )

    fig.update_yaxes(
        tickformat=".0%", title_text="Cumulative Return",
        gridcolor=GRID, zerolinecolor=GRID,
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
        row=1, col=1,
    )
    fig.update_xaxes(
        dtick="M12", tickformat="%Y", tickangle=0,
        showgrid=False, tickfont=dict(color=SUBTEXT),
        row=1, col=1,
    )

    # ── Table: Drawdown Summary ────────────────────────────────────────────────
    headers, cells = _build_drawdown_table(episodes)
    if headers:
        row_colors = [TBL_ROW_B, Q1_COLOR, Q2_COLOR, Q3_COLOR, Q4_COLOR]
        fill_colors = [[row_colors[r] for r in range(len(cells[0]))] for _ in cells]

        fig.add_trace(
            go.Table(
                header=dict(
                    values=[f"<b>{h}</b>" for h in headers],
                    fill_color=TBL_HDR,
                    font=dict(color="white", size=12, family="Inter, Arial"),
                    align="left",
                    height=30,
                    line_color="white",
                ),
                cells=dict(
                    values=cells,
                    fill_color=fill_colors,
                    font=dict(color=TEXT, size=11, family="Inter, Arial"),
                    align="left",
                    height=26,
                    line_color=GRID,
                ),
            ),
            row=2, col=1,
        )

    # ── Chart 3: Underwater ────────────────────────────────────────────────────
    fig.add_trace(
        go.Scatter(
            x=df.index, y=df['drawdown'],
            fill='tozeroy',
            mode='lines',
            line=dict(color=RED, width=1),
            fillcolor='rgba(220,38,38,0.20)',
            name="Drawdown",
            hovertemplate="%{x|%b %d, %Y}<br>Drawdown: %{y:.2%}<extra></extra>",
        ),
        row=3, col=1,
    )
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="x2 domain",
        y0=-threshold, y1=-threshold, yref="y2",
        line=dict(dash="dash", color=SUBTEXT, width=1),
    )
    fig.add_annotation(
        x=1, xref="x2 domain",
        y=-threshold, yref="y2",
        text=f"{threshold:.0%} threshold",
        showarrow=False, xanchor="right", yanchor="top",
        font=dict(color=SUBTEXT, size=10),
    )
    # ── Named crisis bands on drawdown chart ──────────────────────────────────
    for _ep in _named_eps:
        fig.add_shape(
            type='rect',
            x0=_ep['peak_date'], x1=_ep['trough_date'],
            y0=0, y1=1,
            xref='x2', yref='y2 domain',
            fillcolor='rgba(220,38,38,0.30)', opacity=1,
            layer='below', line_width=1.5, line_color='black',
        )
        fig.add_shape(
            type='rect',
            x0=_ep['trough_date'], x1=_ep['recovery_date'],
            y0=0, y1=1,
            xref='x2', yref='y2 domain',
            fillcolor='rgba(22,163,74,0.20)', opacity=1,
            layer='below', line_width=1.5, line_color='black',
        )

    fig.update_yaxes(
        tickformat=".0%", title_text="Drawdown",
        gridcolor=GRID, zerolinecolor=GRID,
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
        row=3, col=1,
    )
    fig.update_xaxes(
        dtick="M12", tickformat="%Y", tickangle=0,
        showgrid=False, tickfont=dict(color=SUBTEXT),
        row=3, col=1,
    )

    # ── Chart 4: Return Distribution ──────────────────────────────────────────
    ret_vals = df['returns_per_day'].dropna()
    neg_vals = ret_vals[ret_vals <  0]
    pos_vals = ret_vals[ret_vals >= 0]
    bin_size = 0.001

    fig.add_trace(
        go.Histogram(
            x=neg_vals, name="Negative",
            xbins=dict(size=bin_size),
            marker_color='rgba(220,38,38,0.70)',
            hovertemplate="Return: %{x:.2%}<br>Count: %{y}<extra></extra>",
        ),
        row=4, col=1,
    )
    fig.add_trace(
        go.Histogram(
            x=pos_vals, name="Positive",
            xbins=dict(size=bin_size),
            marker_color='rgba(22,163,74,0.70)',
            hovertemplate="Return: %{x:.2%}<br>Count: %{y}<extra></extra>",
        ),
        row=4, col=1,
    )
    fig.add_shape(
        type="line",
        x0=0, x1=0, xref="x3",
        y0=0, y1=1, yref="y3 domain",
        line=dict(dash="dot", color=TEXT, width=1),
    )
    mean_ret = float(ret_vals.mean())
    fig.add_shape(
        type="line",
        x0=mean_ret, x1=mean_ret, xref="x3",
        y0=0, y1=1, yref="y3 domain",
        line=dict(dash="dash", color="#D97706", width=1.2),
    )
    fig.add_annotation(
        x=mean_ret, xref="x3",
        y=1, yref="y3 domain",
        text=f"mean {mean_ret:.3%}",
        showarrow=False, xanchor="left", yanchor="top",
        font=dict(color="#D97706", size=10),
    )
    fig.update_xaxes(
        tickformat=".1%", title_text="Daily Return",
        gridcolor=GRID, tickfont=dict(color=SUBTEXT),
        title_font=dict(color=SUBTEXT),
        row=4, col=1,
    )
    fig.update_yaxes(
        title_text="Count", gridcolor=GRID,
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
        row=4, col=1,
    )

    # ── Chart 5: Annual Returns Bar ───────────────────────────────────────────
    bar_colors = [GREEN if v >= 0 else RED for v in df_annual.values]
    fig.add_trace(
        go.Bar(
            x=[str(y) for y in df_annual.index],
            y=df_annual.values,
            marker_color=bar_colors,
            text=[f"{v:.1%}" for v in df_annual.values],
            textposition='outside',
            textfont=dict(size=9, color=TEXT),
            hovertemplate="Year: %{x}<br>Return: %{y:.2%}<extra></extra>",
            name="Annual Return",
        ),
        row=5, col=1,
    )
    fig.add_shape(
        type="line",
        x0=0, x1=1, xref="x4 domain",
        y0=0, y1=0, yref="y4",
        line=dict(color=SUBTEXT, width=1),
    )
    fig.update_yaxes(
        tickformat=".0%", title_text="Return",
        gridcolor=GRID, zerolinecolor=GRID,
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
        row=5, col=1,
    )
    fig.update_xaxes(
        tickfont=dict(color=SUBTEXT), showgrid=False,
        tickangle=-45, row=5, col=1,
    )

    # ── Chart 6: Monthly Heatmap ──────────────────────────────────────────────
    abs_max = float(np.nanmax(np.abs(heat_z)))
    fig.add_trace(
        go.Heatmap(
            z=heat_z,
            x=month_names,
            y=heat_years,
            colorscale=[
                [0.0,  "#DC2626"],
                [0.5,  "#FFFFFF"],
                [1.0,  "#16A34A"],
            ],
            zmid=0,
            zmin=-abs_max,
            zmax=abs_max,
            text=[[f"{v:.1%}" if not np.isnan(v) else "" for v in row] for row in heat_z],
            texttemplate="%{text}",
            textfont=dict(size=9, color=TEXT),
            hovertemplate="Month: %{x}<br>Year: %{y}<br>Return: %{z:.2%}<extra></extra>",
            showscale=True,
            colorbar=dict(
                tickformat=".0%",
                thickness=12,
                len=0.16,
                y=0.05,
                title=dict(text="Return", font=dict(size=10, color=SUBTEXT)),
                tickfont=dict(size=9, color=SUBTEXT),
            ),
        ),
        row=6, col=1,
    )
    fig.update_yaxes(
        title_text="Year", autorange="reversed",
        title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT, size=10),
        row=6, col=1,
    )
    fig.update_xaxes(
        tickfont=dict(color=SUBTEXT), showgrid=False,
        side="bottom", row=6, col=1,
    )

    # ── Subplot title style ────────────────────────────────────────────────────
    for ann in fig['layout']['annotations']:
        if not ann.showarrow:
            ann['font'] = dict(color=TEXT, size=13, family="Inter, Arial")

    # ── Global layout ──────────────────────────────────────────────────────────
    fig.update_layout(
        height=2500,
        barmode='overlay',
        title=dict(
            text=title,
            font=dict(size=20, color=TEXT, family="Inter, Arial"),
            x=0.5, xanchor='center', y=0.99,
        ),
        paper_bgcolor=BG,
        plot_bgcolor=PANEL,
        font=dict(family="Inter, Arial, sans-serif", size=12, color=TEXT),
        showlegend=False,
        margin=dict(l=70, r=50, t=80, b=50),
        hoverlabel=dict(
            bgcolor="white",
            font_size=12,
            font_color=TEXT,
            bordercolor=GRID,
        ),
    )
    fig.update_xaxes(linecolor=GRID, mirror=False)
    fig.update_yaxes(linecolor=GRID, mirror=False)

    return fig.to_html(full_html=False, include_plotlyjs='cdn')


def generate_crisis_comparison_charts(portfolio_df, benchmark_df, title_prefix="") -> str:
    """For each crisis in CRISIS_PERIODS, produce a chart (portfolio vs benchmark,
    normalized to 1.0 at crisis start) and a side-by-side metrics comparison table.
    Returns combined HTML string."""

    BG      = "#FFFFFF"
    PANEL   = "#F8F9FA"
    GRID    = "#E9ECEF"
    TEXT    = "#1E293B"
    SUBTEXT = "#64748B"
    PORT_C  = "#1D6FA4"
    BENCH_C = "#F97316"

    bm_log   = benchmark_df['log_returns_per_day'].sort_index()
    port_log = portfolio_df['log_return'].sort_index()

    divs = []

    def _safe_sortino(lr):
        ann_ret = annualized_return(lr)
        ds_vol  = np.sqrt((np.minimum(lr, 0) ** 2).mean()) * np.sqrt(TRADING_DAYS_PER_YEAR)
        return ann_ret / ds_vol if ds_vol != 0 else np.nan

    def _safe_calmar(lr):
        price = np.exp(lr.cumsum())
        mdd   = abs(maximum_drawdown(price))
        return annualized_return(lr) / mdd if mdd != 0 else np.nan

    for cname, _, c_start, _, c_end in CRISIS_PERIODS:
        p_lr = port_log.loc[c_start:c_end].dropna()
        b_lr = bm_log.loc[c_start:c_end].dropna()

        if p_lr.empty or b_lr.empty:
            divs.append(f'<p style="font-family:Inter,Arial;color:{SUBTEXT};margin:20px 70px">'
                        f'{cname}: insufficient data in portfolio or benchmark.</p>')
            continue

        p_lr, b_lr = p_lr.align(b_lr, join='inner')
        p_cum = np.exp(p_lr.cumsum())
        b_cum = np.exp(b_lr.cumsum())

        def _metrics(lr):
            price = np.exp(lr.cumsum())
            return {
                'Cumulative Return'     : f"{cumulative_return(lr):.2%}",
                'Annualized Return'     : f"{annualized_return(lr):.2%}",
                'Annualized Volatility' : f"{annualized_volatility(lr):.2%}",
                'Max Drawdown'          : f"{maximum_drawdown(price):.2%}",
                'Sharpe Ratio'          : f"{sharpe_ratio(lr, 0.0):.3f}",
                'Sortino Ratio'         : f"{_safe_sortino(lr):.3f}",
                'Calmar Ratio'          : f"{_safe_calmar(lr):.3f}",
                'VaR 95%'               : f"{value_at_risk(lr, 0.95):.2%}",
                'CVaR 95%'              : f"{conditional_value_at_risk(lr, 0.95):.2%}",
                'Omega Ratio'           : f"{omega_ratio(lr):.3f}",
            }

        p_m = _metrics(p_lr)
        b_m = _metrics(b_lr)
        metric_names = list(p_m.keys())

        def _cell_color(name, pval_str, bval_str):
            try:
                pv = float(pval_str.replace('%', ''))
                bv = float(bval_str.replace('%', ''))
            except Exception:
                return TEXT, TEXT
            higher_is_better = name not in ('Annualized Volatility',)
            if pv > bv:
                return ('#166534' if higher_is_better else '#DC2626', TEXT)
            elif pv < bv:
                return ('#DC2626' if higher_is_better else '#166534', TEXT)
            return TEXT, TEXT

        port_colors  = []
        bench_colors = []
        for mn in metric_names:
            pc, bc = _cell_color(mn, p_m[mn], b_m[mn])
            port_colors.append(pc)
            bench_colors.append(bc)

        fig = make_subplots(
            rows=2, cols=1,
            row_heights=[0.55, 0.45],
            vertical_spacing=0.06,
            specs=[[{"type": "xy"}], [{"type": "table"}]],
            subplot_titles=[
                f"{cname}  ·  Cumulative Return  ({c_start} → {c_end})",
                "Metrics Comparison",
            ],
        )

        fig.add_trace(go.Scatter(
            x=p_cum.index, y=p_cum.values,
            mode='lines', name='Portfolio',
            line=dict(color=PORT_C, width=2.5),
            hovertemplate="%{x|%b %d, %Y}<br>Portfolio: %{y:.3f}<extra></extra>",
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=b_cum.index, y=b_cum.values,
            mode='lines', name='Benchmark (S&P 500)',
            line=dict(color=BENCH_C, width=2.5, dash='dash'),
            hovertemplate="%{x|%b %d, %Y}<br>Benchmark: %{y:.3f}<extra></extra>",
        ), row=1, col=1)

        fig.add_shape(
            type='line',
            x0=0, x1=1, xref='x domain',
            y0=1.0, y1=1.0, yref='y',
            line=dict(color=SUBTEXT, width=1, dash='dot'),
        )

        fig.update_yaxes(
            title_text="Normalized Value (1.0 = start)", gridcolor=GRID,
            tickformat=".2f", title_font=dict(color=SUBTEXT), tickfont=dict(color=SUBTEXT),
            row=1, col=1,
        )
        fig.update_xaxes(
            showgrid=False, tickfont=dict(color=SUBTEXT),
            dtick="M1", tickformat="%b %Y", tickangle=-45,
            row=1, col=1,
        )

        fig.add_trace(go.Table(
            header=dict(
                values=["<b>Metric</b>", "<b>Portfolio</b>", "<b>Benchmark (S&P 500)</b>"],
                fill_color='#1D6FA4',
                font=dict(color='white', size=12, family='Inter, Arial'),
                align='left', height=30, line_color='white',
            ),
            cells=dict(
                values=[
                    metric_names,
                    [p_m[mn] for mn in metric_names],
                    [b_m[mn] for mn in metric_names],
                ],
                fill_color=[
                    ['#F8F9FA'] * len(metric_names),
                    ['rgba(29,111,164,0.08)'] * len(metric_names),
                    ['rgba(249,115,22,0.08)'] * len(metric_names),
                ],
                font=dict(
                    color=[
                        [TEXT] * len(metric_names),
                        port_colors,
                        bench_colors,
                    ],
                    size=11, family='Inter, Arial',
                ),
                align='left', height=26, line_color=GRID,
            ),
        ), row=2, col=1)

        for ann in fig['layout']['annotations']:
            ann['font'] = dict(color=TEXT, size=13, family='Inter, Arial')

        fig.update_layout(
            height=800,
            title=dict(
                text=f"{title_prefix}{cname}  ·  Crisis Period Analysis",
                font=dict(size=16, color=TEXT, family='Inter, Arial'),
                x=0.5, xanchor='center',
            ),
            paper_bgcolor=BG,
            plot_bgcolor=PANEL,
            font=dict(family='Inter, Arial, sans-serif', size=12, color=TEXT),
            showlegend=True,
            legend=dict(
                orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1,
                font=dict(color=TEXT, size=11),
            ),
            margin=dict(l=70, r=50, t=80, b=40),
        )
        fig.update_xaxes(linecolor=GRID, mirror=False)
        fig.update_yaxes(linecolor=GRID, mirror=False)

        divs.append(fig.to_html(full_html=False, include_plotlyjs=False))

    return "\n<hr style='border:none;border-top:1px solid #E9ECEF;margin:20px 70px'>\n".join(divs)


# ------
# Helpers: load & average across runs
# ------

def load_and_average_runs(freq_name):
    """Discover all portfolio_rf_{freq}_{run}.csv files, average log_return
    across runs on common dates, recompute cumulative_value. Returns (df, n_runs)."""
    dfs, run = [], 1
    while True:
        p = DATA_DIR / f"portfolio_rf_{freq_name}_{run}.csv"
        if not p.exists():
            break
        df = pd.read_csv(p, index_col='date', parse_dates=True).sort_index()
        df = df.loc[start_date:end_date].dropna(subset=['log_return'])
        dfs.append(df['log_return'])
        run += 1
    if not dfs:
        return None, 0
    avg_log = pd.concat(dfs, axis=1).mean(axis=1).dropna()
    avg_cum = np.exp(avg_log.cumsum())
    avg_df  = pd.DataFrame({'log_return': avg_log, 'cumulative_value': avg_cum})
    return avg_df, len(dfs)


def load_and_average_stats(freq_name):
    """Discover all portfolio_rf_{freq}_{run}_statistics.csv, average per
    rebalance_date. Returns averaged DataFrame or None."""
    dfs, run = [], 1
    STAT_COLS = ['RMSE', 'MSE', 'MAE', 'R_squared', 'Spearman',
                 'Directional_Accuracy', 'Geometric_Score']
    while True:
        p = DATA_DIR / f"portfolio_rf_{freq_name}_{run}_statistics.csv"
        if not p.exists():
            break
        df = pd.read_csv(p, parse_dates=['rebalance_date'])
        dfs.append(df)
        run += 1
    if not dfs:
        return None
    combined = pd.concat(dfs, ignore_index=True)
    existing = [c for c in STAT_COLS if c in combined.columns]
    avg = combined.groupby('rebalance_date')[existing].mean().reset_index()
    return avg


# ------
# Statistics visualisation
# ------

def generate_rf_statistics_report(stats_df, freq_name) -> str:
    """Builds an HTML section with time-series charts + summary table for all
    RF prediction quality metrics (RMSE, MSE, MAE, R², Spearman, DA, Geo-Score)."""
    if stats_df is None or stats_df.empty:
        return '<p style="font-family:Inter,Arial;color:#64748B;margin:20px 70px">No statistics data available.</p>'

    BG      = "#FFFFFF"
    PANEL   = "#F8F9FA"
    GRID    = "#E9ECEF"
    TEXT    = "#1E293B"
    SUBTEXT = "#64748B"

    METRIC_META = {
        'RMSE'                : ('RMSE',                  False, '#1D6FA4'),
        'MSE'                 : ('MSE',                   False, '#7C3AED'),
        'MAE'                 : ('MAE',                   False, '#D97706'),
        'R_squared'           : ('R²',                    True,  '#16A34A'),
        'Spearman'            : ('Spearman Correlation',   True,  '#DC2626'),
        'Directional_Accuracy': ('Directional Accuracy',  True,  '#0891B2'),
        'Geometric_Score'     : ('Geometric Score',        True,  '#059669'),
    }

    available = [c for c in METRIC_META if c in stats_df.columns]
    n = len(available)
    if n == 0:
        return '<p style="font-family:Inter,Arial;color:#64748B;margin:20px 70px">No statistics columns found.</p>'

    # ── Time-series chart (one trace per metric, normalised 0-1 for comparison) ─
    fig_ts = go.Figure()
    for col in available:
        label, _, color = METRIC_META[col]
        s = stats_df.set_index('rebalance_date')[col].dropna()
        fig_ts.add_trace(go.Scatter(
            x=s.index, y=s.values,
            mode='lines+markers', name=label,
            line=dict(color=color, width=1.8),
            marker=dict(size=4),
            hovertemplate=f"%{{x|%Y-%m-%d}}<br>{label}: %{{y:.4f}}<extra></extra>",
        ))

    fig_ts.update_layout(
        title=dict(text=f"RF Prediction Quality Over Time  ·  {freq_name}",
                   font=dict(size=15, color=TEXT, family='Inter, Arial'),
                   x=0.5, xanchor='center'),
        height=420,
        paper_bgcolor=BG, plot_bgcolor=PANEL,
        font=dict(family='Inter, Arial', size=11, color=TEXT),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1,
                    font=dict(size=10)),
        margin=dict(l=60, r=40, t=70, b=40),
        xaxis=dict(showgrid=False, tickfont=dict(color=SUBTEXT)),
        yaxis=dict(gridcolor=GRID, tickfont=dict(color=SUBTEXT)),
    )

    # ── Summary table ─────────────────────────────────────────────────────────
    rows_m, rows_mean, rows_std, rows_min, rows_max = [], [], [], [], []
    for col in available:
        label, _, _ = METRIC_META[col]
        s = stats_df[col].dropna()
        rows_m.append(label)
        rows_mean.append(f"{s.mean():.4f}")
        rows_std.append(f"{s.std():.4f}")
        rows_min.append(f"{s.min():.4f}")
        rows_max.append(f"{s.max():.4f}")

    fig_tbl = go.Figure(go.Table(
        header=dict(
            values=["<b>Metric</b>", "<b>Mean</b>", "<b>Std Dev</b>", "<b>Min</b>", "<b>Max</b>"],
            fill_color='#1D6FA4',
            font=dict(color='white', size=12, family='Inter, Arial'),
            align='left', height=30, line_color='white',
        ),
        cells=dict(
            values=[rows_m, rows_mean, rows_std, rows_min, rows_max],
            fill_color=[['#F8F9FA'] * len(rows_m)] * 5,
            font=dict(color=TEXT, size=11, family='Inter, Arial'),
            align='left', height=26, line_color=GRID,
        ),
    ))
    fig_tbl.update_layout(
        height=60 + 30 * len(rows_m),
        margin=dict(l=60, r=40, t=10, b=10),
        paper_bgcolor=BG,
    )

    ts_html  = fig_ts.to_html(full_html=False, include_plotlyjs=False)
    tbl_html = fig_tbl.to_html(full_html=False, include_plotlyjs=False)
    return ts_html + tbl_html


# ------
# Compute and Export Metrics for Each Rebalancing Frequency
# ------
benchmark_csv = DATA_PATH / "results" / "data" / "benchmark" / "portfolio.csv"
df_bench = pd.read_csv(benchmark_csv, index_col='date', parse_dates=True).sort_index()
df_bench = df_bench.loc[start_date:end_date].dropna(subset=['log_returns_per_day'])

all_divs = []

for freq_name in FREQUENCIES:
    df_port, n_runs = load_and_average_runs(freq_name)
    if df_port is None:
        print(f"Skipping {freq_name}: no run files found.")
        continue
    print(f"[{freq_name}] Loaded & averaged {n_runs} run(s).")

    log_ret   = df_port['log_return']
    price_ser = df_port['cumulative_value']

    results = compute_all_metrics(
        portfolio_log_returns=log_ret,
        price_series=price_ser,
        benchmark_log_returns=df_bench['log_returns_per_day'],
        risk_free_rate=risk_free_rate,
        freq='D',
    )

    df_metrics = metrics_to_dataframe(results)
    metrics_file = output_dir_metrics / f"metrics_{freq_name}.csv"
    df_metrics.to_csv(metrics_file, index=False)
    print(f"[{freq_name}] Metrics exported.")

    # ── Statistics: load, average, export ────────────────────────────────────
    stats_df = load_and_average_stats(freq_name)
    if stats_df is not None:
        stats_file = output_dir_metrics / f"statistics_{freq_name}.csv"
        stats_df.to_csv(stats_file, index=False)
        print(f"[{freq_name}] Statistics exported.")

    run_label = f"avg of {n_runs} run{'s' if n_runs > 1 else ''}"
    section_title = (
        f'<h2 style="font-family:Inter,Arial,sans-serif;color:#1E293B;margin:40px 0 8px 70px">'
        f'Random Forest  ·  {freq_name} Rebalancing  <small style="font-size:0.6em;color:#64748B">({run_label})</small></h2>'
    )

    # Main performance report
    div = generate_dynamic_benchmark_report(
        df_port,
        title=f"Random Forest  ·  {freq_name} Rebalancing  ({run_label})",
        threshold=0.05,
    )

    # RF statistics charts + table
    stats_header = (
        f'<h3 style="font-family:Inter,Arial,sans-serif;color:#1E293B;margin:32px 0 4px 70px">'
        f'Model Prediction Quality  ·  {freq_name}</h3>'
    )
    stats_div = generate_rf_statistics_report(stats_df, freq_name)

    # Crisis comparison charts
    crisis_header = (
        f'<h3 style="font-family:Inter,Arial,sans-serif;color:#1E293B;margin:32px 0 4px 70px">'
        f'Crisis Period Analysis  ·  {freq_name}</h3>'
    )
    try:
        crisis_div = generate_crisis_comparison_charts(
            portfolio_df=df_port,
            benchmark_df=df_bench,
            title_prefix=f"{freq_name}  ·  ",
        )
    except Exception as e:
        print(f"[{freq_name}] Crisis charts error: {e}")
        crisis_div = f'<p style="font-family:Inter,Arial;color:#DC2626;margin:20px 70px">Crisis charts could not be generated: {e}</p>'

    all_divs.append(section_title + div + stats_header + stats_div + crisis_header + crisis_div)
    print(f"[{freq_name}] All charts built.")

# Write single combined HTML
combined_report_path = output_dir_plots / "report_all_frequencies.html"
combined_html = """<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Random Forest Portfolio — All Frequencies</title>
</head>
<body style="margin:0;background:#ffffff">
""" + "\n<hr style='border:none;border-top:2px solid #E9ECEF;margin:0 70px'>\n".join(all_divs) + """
</body>
</html>"""

combined_report_path.write_text(combined_html, encoding='utf-8')
print(f"\nCombined report exported to: {combined_report_path}")

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

_DATA_PATH = Path(r"C:/Users/benel/OneDrive/Desktop/Python/Thesis_xyz")
_DATA_DIR  = _DATA_PATH / "results" / "data"
_START, _END = "1998-01-01", "2025-12-31"

CRISIS_PERIODS = [
    ("Dotcom Crash",       "2000-03-23", "2007-05-31"),
    ("GFC",                "2007-10-09", "2013-03-28"),
    ("Monetary Policy",    "2018-09-21", "2019-04-23"),
    ("COVID-19",           "2020-02-19", "2020-08-12"),
    ("Russia/Ukraine",     "2022-01-03", "2024-01-19"),
    ("Trade Policy Shock", "2025-02-19", "2025-06-26"),
]
BG, PANEL, GRID = "#FFFFFF", "#F8F9FA", "#E9ECEF"
TEXT, SUBTEXT   = "#1E293B", "#64748B"

def _avg_runs(model_dir, pattern):
    dfs, run = [], 1
    while True:
        p = model_dir / pattern.format(run=run)
        if not p.exists(): break
        df = pd.read_csv(p, index_col="date", parse_dates=True).sort_index()
        df = df.loc[_START:_END].dropna(subset=["log_return"])
        dfs.append(df["log_return"])
        run += 1
    if not dfs: return None, 0
    avg = pd.concat(dfs, axis=1).mean(axis=1).dropna()
    return pd.DataFrame({"log_return": avg, "cumulative_value": np.exp(avg.cumsum())}), len(dfs)

def _single(path, log_col="log_return"):
    if not path.exists(): return None
    df = pd.read_csv(path, index_col="date", parse_dates=True).sort_index()
    df = df.loc[_START:_END].dropna(subset=[log_col])
    if log_col != "log_return":
        df = df.rename(columns={log_col: "log_return"})
    df["cumulative_value"] = np.exp(df["log_return"].cumsum())
    return df

rf_df,  rf_n   = _avg_runs(_DATA_DIR / "random_forest", "portfolio_rf_Monthly_{run}.csv")
xgb_df, xgb_n  = _avg_runs(_DATA_DIR / "xgboost",      "portfolio_xgb_Monthly_{run}.csv")
ew_df          = _single(_DATA_DIR / "equal_weight" / "portfolio_Monthly.csv")
mz_df          = _single(_DATA_DIR / "markowitz"    / "portfolio_Monthly.csv")
bench_df       = _single(_DATA_DIR / "benchmark"    / "portfolio.csv", log_col="log_returns_per_day")

series = {}
if rf_df    is not None: series[f"Random Forest Monthly (avg {rf_n} runs)"] = (rf_df["cumulative_value"],    "#1D6FA4")
if xgb_df   is not None: series[f"XGBoost Monthly (avg {xgb_n} runs)"]      = (xgb_df["cumulative_value"],   "#0891B2")
if ew_df    is not None: series["Equal Weight Monthly"]                       = (ew_df["cumulative_value"],    "#16A34A")
if mz_df    is not None: series["Markowitz Monthly"]                          = (mz_df["cumulative_value"],    "#7C3AED")
if bench_df is not None: series["S&P 500 (Benchmark)"]                        = (bench_df["cumulative_value"], "#F97316")

fig = go.Figure()

if series:
    first_idx = next(iter(series.values()))[0].index
    for cname, c_start, c_end in CRISIS_PERIODS:
        s = first_idx[first_idx >= pd.Timestamp(c_start)]
        e = first_idx[first_idx <= pd.Timestamp(c_end)]
        if s.empty or e.empty: continue
        fig.add_vrect(x0=s[0], x1=e[-1], fillcolor="rgba(220,38,38,0.10)", layer="below", line_width=0)
        fig.add_annotation(
            x=s[0], xref="x", y=1.01, yref="paper",
            text=cname, showarrow=False, xanchor="left", yanchor="bottom",
            font=dict(size=9, color="#DC2626", family="Inter, Arial"),
        )

for label, (cum_series, color) in series.items():
    fig.add_trace(go.Scatter(
        x=cum_series.index, y=cum_series.values,
        mode="lines", name=label,
        line=dict(color=color, width=2),
        hovertemplate="%{x|%b %d, %Y}<br>" + label + ": %{y:.3f}x<extra></extra>",
    ))

fig.update_layout(
    title=dict(text="Cumulative Return  —  Log Scale", font=dict(size=18, color=TEXT, family="Inter, Arial"), x=0.5, xanchor="center"),
    height=580, paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(family="Inter, Arial, sans-serif", size=12, color=TEXT),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, font=dict(color=TEXT, size=11)),
    margin=dict(l=80, r=50, t=100, b=50),
    hoverlabel=dict(bgcolor="white", font_size=12, font_color=TEXT, bordercolor=GRID),
)
fig.update_xaxes(showgrid=False, tickfont=dict(color=SUBTEXT), linecolor=GRID)
fig.update_yaxes(
    type="log", tickformat=".2f", ticksuffix="x",
    gridcolor=GRID, tickfont=dict(color=SUBTEXT),
    title_text="Portfolio Value (log scale, start = 1.0)",
    title_font=dict(color=SUBTEXT), linecolor=GRID,
)
fig.show()
